**ECE 447: Data Analysis and Machine Learning for Engineers**

# Assignment 4


In [ ]:
import numpy as np
import torch
import wandb
import matplotlib.pyplot as plt

In [ ]:
# wandb logging
wandb_flag = False

def train_test_split(X, y, test_ratio=0.25):
    n = X.shape[0]
    idx = torch.randperm(n)
    n_test = int(round(test_ratio * n))
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def accuracy(y_true, y_pred):
    return (y_true == y_pred).float().mean().item()

def confusion_matrix(y_true, y_pred):
    # y_true, y_pred in {0,1}
    TP = int(((y_true == 1) & (y_pred == 1)).sum().item())
    TN = int(((y_true == 0) & (y_pred == 0)).sum().item())
    FP = int(((y_true == 0) & (y_pred == 1)).sum().item())
    FN = int(((y_true == 1) & (y_pred == 0)).sum().item())
    return {"TP": TP, "TN": TN, "FP": FP, "FN": FN}

def plot_points_2d(X, y, title="Data"):
    Xn = X.detach().cpu().numpy()
    yn = y.detach().cpu().numpy()
    plt.figure()
    plt.scatter(Xn[yn==0,0], Xn[yn==0,1], s=10, label="class 0")
    plt.scatter(Xn[yn==1,0], Xn[yn==1,1], s=10, label="class 1")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.axis('equal')
    plt.tight_layout()
    plt.legend();

def make_grid_2d(X, pad=1.0, n_grid=250):
    Xn = X.detach().cpu().numpy()
    x1_min, x1_max = Xn[:,0].min() - pad, Xn[:,0].max() + pad
    x2_min, x2_max = Xn[:,1].min() - pad, Xn[:,1].max() + pad
    xs = np.linspace(x1_min, x1_max, n_grid)
    ys = np.linspace(x2_min, x2_max, n_grid)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.stack([xx.ravel(), yy.ravel()], axis=1)
    return torch.tensor(grid, dtype=torch.float32), xx, yy

def plot_decision_boundary(X, y, predict_fn, title="Decision boundary"):
    grid, xx, yy = make_grid_2d(X)
    with torch.no_grad():
        pred = predict_fn(grid).detach().cpu().numpy().reshape(xx.shape)
    plt.figure()
    plot_points_2d(X, y, title=title)
    plt.contourf(xx, yy, pred, alpha=0.25, levels=[-0.5,0.5,1.5])
    plt.show();
    
def sample_gaussian(mean, cov, n, device="cpu"):
    dist = torch.distributions.MultivariateNormal(mean, covariance_matrix=cov)
    return dist.sample((n,))

def generate_two_class_data(mu0, Sigma0, mu1, Sigma1, n0=500, n1=500, device="cpu"):
    X0 = sample_gaussian(mu0, Sigma0, n0, device=device)
    X1 = sample_gaussian(mu1, Sigma1, n1, device=device)
    y0 = torch.zeros(n0, dtype=torch.long, device=device)
    y1 = torch.ones(n1)
    X = torch.cat([X0, X1], dim=0)
    y = torch.cat([y0, y1], dim=0)
    return X, y

def ml_mean_cov(X):
    mu_hat = X.mean(dim=0)
    Xc = X - mu_hat
    Sigma_hat = (Xc.T @ Xc) / X.shape[0]
    return mu_hat, Sigma_hat

def classwise_ml_estimates(X, y):
    mu0, S0 = ml_mean_cov(X[y==0])
    mu1, S1 = ml_mean_cov(X[y==1])
    return (mu0, S0), (mu1, S1)

In [ ]:
##### Here are wrote the general pdfs to compute the boundary.
##### Alternatively, you can implement the derived equation asked in the assignment.

# ----------------------------
# MAP classifier (QDA with Gaussians)
# ----------------------------
def log_gaussian_pdf(x, mu, Sigma, eps=1e-6):
    d = x.shape[1]
    Sigma_reg = Sigma + eps * torch.eye(d, device=x.device)
    L = torch.linalg.cholesky(Sigma_reg)
    diff = x - mu
    # Solve Sigma^{-1} diff using Cholesky
    sol = torch.cholesky_solve(diff.unsqueeze(-1), L).squeeze(-1)
    quad = (diff * sol).sum(dim=1)  # (N,)
    logdet = 2.0 * torch.log(torch.diagonal(L)).sum()
    return -0.5 * (quad + logdet + d * np.log(2 * np.pi))

def map_predict_fn(mu0, S0, mu1, S1, prior0=0.5, prior1=0.5):
    logp0 = np.log(prior0)
    logp1 = np.log(prior1)

    def predict(X):
        l0 = log_gaussian_pdf(X, mu0, S0) + logp0
        l1 = log_gaussian_pdf(X, mu1, S1) + logp1
        return (l1 > l0).long()  # 1 if class1 more likely
    return predict

# ----------------------------
# MED classifier
# ----------------------------
def med_predict_fn(mu0, mu1):
    def predict(X):
        d0 = ((X - mu0) ** 2).sum(dim=1)
        d1 = ((X - mu1) ** 2).sum(dim=1)
        return (d1 < d0).long()
    return predict

# ----------------------------
# Polynomial features (2D -> poly)
# ----------------------------
def poly_features_2d(X, degree):
    """
    For X = [x1, x2], returns [1, x1, x2, x1^2, x1 x2, x2^2, ...] up to 'degree'.
    """
    x1 = X[:, 0:1]
    x2 = X[:, 1:2]
    feats = [torch.ones_like(x1)]
    for d in range(1, degree + 1):
        for i in range(d + 1):
            j = d - i
            feats.append((x1 ** i) * (x2 ** j))
    return torch.cat(feats, dim=1)

# ----------------------------
# Logistic regression (PyTorch)
# ----------------------------
class LogisticRegression(torch.nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.linear = torch.nn.Linear(in_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)  # logits

In [ ]:
def train_logreg(
    X_train, y_train, X_test, y_test,
    lr=1e-2, epochs=300, batch_size=128,
    l2=0.0, l1=0.0,
    wandb_flag=False, run_name="logreg"
):
    device = X_train.device
    model = LogisticRegression(X_train.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)  # L2 via weight_decay
    loss_fn = torch.nn.BCEWithLogitsLoss()

    train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_train, y_train.float()), batch_size=batch_size, shuffle=True)

    if wandb_flag:
        wandb.init(project="ece447-a4", name=run_name, config={
            "lr": lr, "epochs": epochs, "batch_size": batch_size, "l2": l2, "l1": l1,
            "in_dim": X_train.shape[1]
        })

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:
            opt.zero_grad()
            logits = model(xb)
            base_loss = loss_fn(logits, yb)

            # L1 penalty on weights (not bias)
            l1_pen = 0.0
            if l1 > 0:
                l1_pen = l1 * model.linear.weight.abs().sum()

            loss = base_loss + l1_pen
            loss.backward()
            opt.step()
            total_loss += loss.item() * xb.shape[0]

        # Evaluate
        model.eval()
        with torch.no_grad():
            train_logits = model(X_train)
            test_logits = model(X_test)
            train_pred = (torch.sigmoid(train_logits) >= 0.5).long()
            test_pred = (torch.sigmoid(test_logits) >= 0.5).long()

            train_acc = accuracy(y_train, train_pred)
            test_acc = accuracy(y_test, test_pred)

            w = model.linear.weight.detach().view(-1)
            w_l1 = w.abs().sum().item()
            w_l2 = torch.sqrt((w**2).sum()).item()

        if wandb_flag:
            wandb.log({
                "epoch": epoch,
                "loss": total_loss / X_train.shape[0],
                "train_acc": train_acc,
                "test_acc": test_acc,
                "w_l1": w_l1,
                "w_l2": w_l2,
            })

    # Final metrics
    cm_train = confusion_matrix(y_train, train_pred)
    cm_test = confusion_matrix(y_test, test_pred)

    return model, {"train_acc": train_acc, "test_acc": test_acc, "cm_train": cm_train, "cm_test": cm_test}

def plot_logreg_boundary_2d(model, feature_fn, X, y, title="LogReg boundary"):
    def predict_fn(grid):
        feats = feature_fn(grid)
        logits = model(feats)
        return (torch.sigmoid(logits) >= 0.5).long()
    plot_decision_boundary(X, y, predict_fn, title=title)


In [ ]:
def run_logistic_regression(X, y, poly_degree=None, l1=0.0, l2=0.0, lr=1e-2, epochs=300, device="cpu", data_name=None):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_ratio=0.25)

    # Feature fn for LR
    if poly_degree is None:
        feature_fn = lambda Z: Z
        Xtr = X_train
        Xte = X_test
        in_dim = 2
        title_suffix = "linear features"
    else:
        feature_fn = lambda Z: poly_features_2d(Z, poly_degree)
        Xtr = feature_fn(X_train)
        Xte = feature_fn(X_test)
        in_dim = Xtr.shape[1]
        title_suffix = f"poly degree {poly_degree}"

    run_name = f"logreg_{title_suffix}_l1={l1}_l2={l2}"
    if data_name:
        run_name = f"{data_name}_{run_name}"


    model, metrics = train_logreg(
        Xtr, y_train, Xte, y_test,
        lr=lr, epochs=epochs,
        l1=l1, l2=l2,
        wandb_flag=wandb_flag, run_name=run_name
    )

    print("Train acc:", metrics["train_acc"], "Test  acc:", metrics["test_acc"])
    print("Weight vector:", model.linear.weight.detach().cpu().numpy().ravel(), "bias:", model.linear.bias.detach().cpu().numpy().ravel())
    # Plot learned boundary in original input space
    plot_logreg_boundary_2d(model, feature_fn, X, y, title=f"LogReg boundary ({title_suffix})")

    return model, metrics

In [ ]:
device = "cpu"
epochs = 450

mu0 = torch.tensor([1.0, 5.0], dtype=torch.float32, device=device)
mu1 = torch.tensor([1.0, 5.0], dtype=torch.float32, device=device)
S0  = torch.tensor([[1.0, 0.0], [0.0, 5.0]], dtype=torch.float32, device=device)
S1  = torch.tensor([[5.0, 0.0], [0.0, 1.0]], dtype=torch.float32, device=device)


m0_err = list()
m1_err = list()
S0_err = list()
S1_err = list()

n_list = np.linspace(10,50000,100)
for i in n_list:
       n = int(i)
       # (a) data
       X, y = generate_two_class_data(mu0, S0, mu1, S1, n, n, device=device)

       # (b) ML estimates
       (mu0_hat, S0_hat), (mu1_hat, S1_hat) = classwise_ml_estimates(X, y)

       print("ML estimates:")
       print("mu0_hat:", mu0_hat.cpu())
       print("S0_hat:", S0_hat.cpu())
       print("mu1_hat:", mu1_hat.cpu())
       print("S1_hat:", S1_hat.cpu())

       m0_err.append(np.sqrt(((mu0 - mu0_hat.cpu())**2).sum().item()))
       m1_err.append(np.sqrt(((mu1 - mu1_hat.cpu())**2).sum().item()))
       S0_err.append(np.sqrt(((S0 - S0_hat.cpu())**2).sum().item()))
       S1_err.append(np.sqrt(((S1 - S1_hat.cpu())**2).sum().item()))

plt.figure()
plt.plot(n_list, m0_err, label="m0 err")
plt.plot(n_list, m1_err, label="m1 err")
plt.plot(n_list, 1/np.sqrt(n_list), label="1/sqrt(n)")
plt.plot(n_list, 1/n_list, label="1/n")
plt.xlabel("n")
plt.ylabel("m ML estimates error")
plt.yscale("log")
plt.legend()

plt.figure()
plt.plot(n_list, S0_err, label="S0 err")
plt.plot(n_list, S1_err, label="S1 err")
plt.plot(n_list, 1/np.sqrt(n_list), label="1/sqrt(n)")
plt.plot(n_list, 1/n_list, label="1/n")
plt.xlabel("n")
plt.ylabel("S ML estimates error")
plt.yscale("log")
plt.legend()

# Here I have also plotted 1/n and 1/sqrt(n) for you to see at which rate the error is decreasing.

In [ ]:
def run_setting(mu0, S0, mu1, S1, n0, n1, epochs, l1=1e-3, l2=1e-3, data_name=None, device="cpu"):

    # (a)  
    X, y = generate_two_class_data(mu0, S0, mu1, S1, n0, n1, device=device)

    # (b)
    (mu0_hat, S0_hat), (mu1_hat, S1_hat) = classwise_ml_estimates(X, y)

    # (d) MAP boundary
    print("\n (d) MAP")
    prior0 = n0 / (n0 + n1)
    prior1 = 1 - prior0
    map_pred = map_predict_fn(mu0_hat, S0_hat, mu1_hat, S1_hat, prior0=prior0, prior1=prior1)
    plot_decision_boundary(X, y, map_pred, title="MAP decision boundary with Gaussian assumption")

    # (e) MED boundary (using true means)
    print("\n (e) MED")
    med_pred = med_predict_fn(mu0_hat, mu1_hat)
    plot_decision_boundary(X, y, med_pred, title="MED decision boundary")

    # (f)
    print("\n (f) LR")
    run_logistic_regression(X, y, poly_degree=None, l1=0.0, l2=0.0, epochs=epochs, device=device, data_name=data_name)

    # (g) L2 regularization example
    print("\n (g) LR with L2 reg")
    run_logistic_regression(X, y, poly_degree=None, l1=0.0, l2=l2, epochs=epochs, device=device, data_name=data_name)

    # (g) L1 regularization example
    print("\n (g) LR with L1 reg")
    run_logistic_regression(X, y, poly_degree=None, l1=l1, l2=0.0, epochs=epochs, device=device, data_name=data_name)

    # (h) polynomial
    print("\n (h) polynomial")
    for deg in [1, 2, 3, 4, 5, 6]:
        print("degree:", deg)
        run_logistic_regression(X, y, poly_degree=deg, l1=0.0, l2=0.0, epochs=epochs, device=device, data_name=data_name)

        print("degree:", deg, "with l1")
        run_logistic_regression(X, y, poly_degree=deg, l1=l1, l2=0.0, epochs=epochs, device=device, data_name=data_name)

        print("degree:", deg, "with l2")
        run_logistic_regression(X, y, poly_degree=deg, l1=0.0, l2=l2, epochs=epochs, device=device, data_name=data_name)


In [ ]:
mu0 = torch.tensor([1.0, 5.0], dtype=torch.float32, device=device)
mu1 = torch.tensor([1.0, 5.0], dtype=torch.float32, device=device)
S0  = torch.tensor([[1.0, 0.0], [0.0, 5.0]], dtype=torch.float32, device=device)
S1  = torch.tensor([[5.0, 0.0], [0.0, 1.0]], dtype=torch.float32, device=device)


n0 = 500
n1 = 500
device = "cpu"
epochs = 450
l1 = 1e-2
l2 = 1e-2

run_setting(mu0, S0, mu1, S1, n0, n1, epochs, l1, l2, "samemean", device)

In [ ]:
mu0 = torch.tensor([1.0, 5.0], dtype=torch.float32, device=device)
mu1 = torch.tensor([1.0, 5.0], dtype=torch.float32, device=device)
S0  = torch.tensor([[10.0, 0.0], [0.0, 4.0]], dtype=torch.float32, device=device)
S1  = torch.tensor([[1.0, 0.0], [0.0, 1.0]], dtype=torch.float32, device=device)

n0 = 500
n1 = 500
device = "cpu"
epochs = 450
l1 = 1e-2
l2 = 1e-2

run_setting(mu0, S0, mu1, S1, n0, n1, epochs, l1, l2, "inside", device)

In [ ]:
mu0 = torch.tensor([-1.0, 10.0], dtype=torch.float32, device=device)
mu1 = torch.tensor([1.0, 5.0], dtype=torch.float32, device=device)
S0  = torch.tensor([[6.0, -2.0], [-2.0, 3.0]], dtype=torch.float32, device=device)
S1  = torch.tensor([[1.0, 0.0], [0.0, 2.0]], dtype=torch.float32, device=device)


n0 = 500
n1 = 500
device = "cpu"
epochs = 450
l1 = 1e-2
l2 = 1e-2

run_setting(mu0, S0, mu1, S1, n0, n1, epochs, l1, l2, "diffmean", device)